<a href="https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmed196/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For this assignment, one row in the source fact table represents one content page for one report date. I will use the March 2026 mid-panel window, from 2026-03-01 through 2026-03-31, rather than the final June 2026 month. My lane will use fact_content_daily_performance for daily search-performance measurements, dim_content for page-level content attributes, and dim_clients for client access and availability context. For modelling, these daily records will later be aggregated into one row per content page for the selected month so that each page can receive one refresh-priority score.

In [1]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

assert hf_token is not None, "HF_TOKEN secret not found."
from huggingface_hub import hf_hub_download
import pandas as pd

repo_id = "FlyRank/internship-warehouse"

files_to_inspect = {
    "clients": "dim_clients.parquet",
    "content": "dim_content.parquet",
    "march_2026": (
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    ),
}

tables = {}

for table_name, file_path in files_to_inspect.items():
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_path,
        repo_type="dataset",
        token=hf_token
    )

    tables[table_name] = pd.read_parquet(local_path)

    print(f"\n--- {table_name} ---")
    print("Rows:", tables[table_name].shape[0])
    print("Columns:", tables[table_name].shape[1])
    print("Column names:")
    print(tables[table_name].columns.tolist())


--- clients ---
Rows: 104
Columns: 9
Column names:
['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']

--- content ---
Rows: 519606
Columns: 26
Column names:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

--- march_2026 ---
Rows: 9841378
Columns: 30
Column names:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_av

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

For my Content Refresh Prioritization lane, I will organize the fields as follows:

Features:
1. gsc_impressions — page visibility during the selected month.
2. gsc_clicks — observed search clicks during the selected month.
3. gsc_avg_position — the page's average Google Search position.
4. word_count — a page-level content attribute from dim_content.
5. search_volume — estimated keyword demand associated with the content page.

Label / proxy:
The label proxy will represent whether the page subsequently shows declining search performance. It will be derived from a future outcome window, not from the same March feature window. The exact label calculation must not be included among the input features.

Context:
report_date, client_hash_id, and content_hash_id will be used for time filtering, joining tables, grouping daily rows into one monthly page record, and client-holdout validation. They will not be treated as predictive numeric features.

Excluded:
I will exclude raw identifiers such as keyword_hash_id and url_hash_id from the model because they identify records rather than generalizable behaviour. I will also exclude any future-period outcome, label-derived field, post-decision measurement, or direct traffic-trend answer because these would create data leakage. Rows without confirmed required data availability will be filtered using the relevant availability flag with IS TRUE.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# March 2026 dataframe ko DuckDB SQL table ke taur par register karo
con = duckdb.connect()
con.register("march_2026", tables["march_2026"])
con.register("dim_content", tables["content"])

print("QUERY 1 — Grain verification")

grain_check = con.execute("""
    WITH page_day_counts AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS rows_per_page_day
        FROM march_2026
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
    )
    SELECT
        SUM(rows_per_page_day) AS total_rows,
        COUNT(*) AS unique_page_day_keys,
        SUM(rows_per_page_day - 1) AS duplicate_rows,
        MAX(rows_per_page_day) AS maximum_rows_per_page_day
    FROM page_day_counts
""").df()

display(grain_check)
print("QUERY 2 — Slice size and date span")

slice_check = con.execute("""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_page_count,
        MIN(report_date) AS minimum_report_date,
        MAX(report_date) AS maximum_report_date
    FROM march_2026
""").df()

display(slice_check)
print("QUERY 3 — GSC availability check")

availability_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS rows_with_gsc_data,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS NOT TRUE
        ) AS rows_without_confirmed_gsc_data,
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) / COUNT(*),
            2
        ) AS percent_rows_with_gsc_data
    FROM march_2026
""").df()

display(availability_check)

QUERY 1 — Grain verification


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_page_day_keys,duplicate_rows,maximum_rows_per_page_day
0,9841378.0,9841378,0.0,1


QUERY 2 — Slice size and date span


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,client_count,content_page_count,minimum_report_date,maximum_report_date
0,9841378,55,331437,2026-03-01,2026-03-31


QUERY 3 — GSC availability check


,total_rows,rows_with_gsc_data,rows_without_confirmed_gsc_data,percent_rows_with_gsc_data
0,9841378,3611061,6230317,36.69


In [3]:
# Build the final March 2026 page-level feature frame
march_page_features = con.execute("""
    WITH monthly_gsc AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS gsc_impressions,
            SUM(gsc_clicks) AS gsc_clicks,
            AVG(gsc_avg_position) AS gsc_avg_position
        FROM march_2026
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,

        -- Exactly five model features
        m.gsc_impressions,
        m.gsc_clicks,
        m.gsc_avg_position,
        c.word_count,
        c.search_volume

    FROM monthly_gsc AS m

    INNER JOIN dim_content AS c
        ON m.client_hash_id = c.client_hash_id
       AND m.content_hash_id = c.content_hash_id

    WHERE c.is_published IS TRUE
      AND c.is_deleted IS NOT TRUE
""").df()

feature_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "search_volume"
]

print("Feature-frame shape:", march_page_features.shape)

print("\nMissing values:")
display(
    march_page_features[feature_columns]
    .isna()
    .sum()
    .to_frame("missing_count")
)

display(march_page_features.head(10))

Feature-frame shape: (176568, 7)

Missing values:


,missing_count
gsc_impressions,0
gsc_clicks,0
gsc_avg_position,0
word_count,55174
search_volume,15780


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,search_volume
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,NaN,20.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,NaN,10.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,NaN,90.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,NaN,10.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,2475.0,50.0
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,48.0,0.0,14.753175,NaN,10.0
6,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,6.341880,NaN,30.0
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,0.0,12.791667,NaN,110.0
8,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,4.950311,2809.0,30.0
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,50.390299,NaN,1900.0


The March 2026 fact slice contains 9,841,378 rows across 55 clients and 331,437 content pages. The date range is complete from 2026-03-01 through 2026-03-31. The combination of report_date, client_hash_id, and content_hash_id is unique, confirming that one raw row represents one content page on one reporting date. Only 3,611,061 rows, or 36.69%, have confirmed GSC availability, so later feature construction will retain rows where gsc_data_available IS TRUE.

Feature availability at the March 2026 decision moment:

1. gsc_impressions — knowable at the decision moment because it is aggregated only from confirmed GSC observations dated 2026-03-01 through 2026-03-31.
2. gsc_clicks — knowable at the decision moment because it is aggregated only from confirmed GSC observations within the selected March window.
3. gsc_avg_position — knowable at the decision moment because it is calculated only from March 2026 search-position records available by 2026-03-31.
4. word_count — knowable at the decision moment because it is a stored page-level content attribute in dim_content rather than a future performance outcome.
5. search_volume — knowable at the decision moment because it is an existing keyword-demand estimate stored in dim_content and is not derived from the future label window.

The identifiers client_hash_id and content_hash_id are retained only for joining, grouping, and client-holdout validation. They are not model features.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# Load April 2026 as the future outcome window
april_file = (
    "fact_content_daily_performance/"
    "month=2026-04/data_0.parquet"
)

april_local_path = hf_hub_download(
    repo_id=repo_id,
    filename=april_file,
    repo_type="dataset",
    token=hf_token
)

tables["april_2026"] = pd.read_parquet(april_local_path)

print("April rows:", tables["april_2026"].shape[0])
print("April columns:", tables["april_2026"].shape[1])
print(
    "Date range:",
    tables["april_2026"]["report_date"].min(),
    "to",
    tables["april_2026"]["report_date"].max()
)

April rows: 10424730
April columns: 30
Date range: 2026-04-01 to 2026-04-30


In [5]:
# Register April table in DuckDB
con.register("april_2026", tables["april_2026"])

# Build a future-outcome proxy using average daily impressions.
# This avoids comparing a 31-day March total against a 30-day April total.

model_frame = con.execute("""
    WITH march_monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS march_avg_daily_impressions
        FROM march_2026
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    april_monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS april_avg_daily_impressions
        FROM april_2026
        WHERE gsc_data_available IS TRUE
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,

        f.gsc_impressions,
        f.gsc_clicks,
        f.gsc_avg_position,
        f.word_count,
        f.search_volume,

        m.march_avg_daily_impressions,
        a.april_avg_daily_impressions,

        CASE
            WHEN a.april_avg_daily_impressions
                 < m.march_avg_daily_impressions
            THEN 1
            ELSE 0
        END AS is_declining_target

    FROM march_page_features AS f

    INNER JOIN march_monthly AS m
        ON f.client_hash_id = m.client_hash_id
       AND f.content_hash_id = m.content_hash_id

    INNER JOIN april_monthly AS a
        ON f.client_hash_id = a.client_hash_id
       AND f.content_hash_id = a.content_hash_id
""").df()

print("Model-frame shape:", model_frame.shape)

print("\nTarget distribution:")
display(
    model_frame["is_declining_target"]
    .value_counts(dropna=False)
    .rename_axis("is_declining_target")
    .to_frame("row_count")
)

print(
    "\nDeclining rate:",
    round(model_frame["is_declining_target"].mean(), 3)
)

display(model_frame.head(10))

Model-frame shape: (158464, 10)

Target distribution:


,row_count
is_declining_target,
1,95338
0,63126



Declining rate: 0.602


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,word_count,search_volume,march_avg_daily_impressions,april_avg_daily_impressions,is_declining_target
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,2123.0,20.0,210.419355,226.233333,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,NaN,40.0,14.612903,13.500000,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,2546.0,10.0,181.612903,282.500000,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2330.0,10.0,159.483871,203.033333,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,NaN,50.0,13.838710,9.566667,1
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,9.445635,NaN,40.0,7.193548,18.400000,0
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,6.014516,NaN,70.0,3.096774,3.827586,0
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,9.155335,NaN,40.0,10.129032,10.733333,0
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,5.258331,2556.0,720.0,248.677419,175.366667,1
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,3561.0,10.0,8.834415,3010.0,70.0,114.870968,139.900000,0


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score

honest_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "word_count",
    "search_volume"
]

target_column = "is_declining_target"

# Deliberate leakage: this column directly copies the future label
model_frame["future_decline_answer"] = model_frame[target_column]

# Client-holdout split so the same client does not appear in train and test
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    splitter.split(
        model_frame,
        model_frame[target_column],
        groups=model_frame["client_hash_id"]
    )
)

train_df = model_frame.iloc[train_index].copy()
test_df = model_frame.iloc[test_index].copy()

def build_model():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=6,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

# 1. Leaky model
leaky_features = honest_features + ["future_decline_answer"]

leaky_model = build_model()
leaky_model.fit(
    train_df[leaky_features],
    train_df[target_column]
)

leaky_predictions = leaky_model.predict(test_df[leaky_features])

leaky_accuracy = accuracy_score(
    test_df[target_column],
    leaky_predictions
)

leaky_precision = precision_score(
    test_df[target_column],
    leaky_predictions,
    zero_division=0
)
# Remove the deliberately leaked field after demonstrating its effect
model_frame.drop(columns=["future_decline_answer"], inplace=True)

print(
    "\nLeakage column still present:",
    "future_decline_answer" in model_frame.columns
)

# 2. Honest model after removing leaked feature
honest_model = build_model()
honest_model.fit(
    train_df[honest_features],
    train_df[target_column]
)

honest_predictions = honest_model.predict(test_df[honest_features])

honest_accuracy = accuracy_score(
    test_df[target_column],
    honest_predictions
)

honest_precision = precision_score(
    test_df[target_column],
    honest_predictions,
    zero_division=0
)

print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

print("\nLEAKY MODEL")
print("Accuracy:", round(leaky_accuracy, 3))
print("Precision:", round(leaky_precision, 3))

print("\nHONEST MODEL")
print("Accuracy:", round(honest_accuracy, 3))
print("Precision:", round(honest_precision, 3))



Leakage column still present: False
Train clients: 36
Test clients: 9

LEAKY MODEL
Accuracy: 1.0
Precision: 1.0

HONEST MODEL
Accuracy: 0.685
Precision: 0.686


I deliberately added a label-derived feature called future_decline_answer, which directly copied is_declining_target. With this leaked feature included, the client-holdout model achieved 1.000 accuracy and 1.000 precision. This near-perfect result is not genuine predictive performance because the model was given the future answer as an input.

After removing future_decline_answer and retaining only the five decision-time features, the honest model achieved 0.684 accuracy and 0.687 precision. The lower score is more credible because it reflects performance using information that would actually be available at the March 2026 decision moment.

Named limitation:
The label is only a proxy based on whether average daily impressions in April are lower than average daily impressions in March. A decline may be caused by seasonality, demand changes, tracking differences, or other external factors rather than content quality alone. In addition, word_count and search_volume contain missing values, so median imputation may hide meaningful differences in data availability.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.